In [1]:
library(DESeq2)
library(dplyr)
library('Matrix')
library(tibble)
library(zoo)
library(stringr)
library(lubridate)
library(fgsea)
library(stringr)
library(gage)
library(devtools)
library(pals)
library(ggplot2)
library(repr)
library(ggrepel)

Loading required package: S4Vectors

Loading required package: stats4

Loading required package: BiocGenerics


Attaching package: ‘BiocGenerics’


The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs


The following objects are masked from ‘package:base’:

    anyDuplicated, aperm, append, as.data.frame, basename, cbind,
    colnames, dirname, do.call, duplicated, eval, evalq, Filter, Find,
    get, grep, grepl, intersect, is.unsorted, lapply, Map, mapply,
    match, mget, order, paste, pmax, pmax.int, pmin, pmin.int,
    Position, rank, rbind, Reduce, rownames, sapply, setdiff, sort,
    table, tapply, union, unique, unsplit, which.max, which.min



Attaching package: ‘S4Vectors’


The following object is masked from ‘package:utils’:

    findMatches


The following objects are masked from ‘package:base’:

    expand.grid, I, unname


Loading required package: IRanges

Loading required package: GenomicRanges

Loading required package: GenomeInfoDb

Loa

In [5]:
local_corr_modules = read.csv('/data/chanjlab/CRC_ZFP36L2.092023/Patients.HTAN/notebooks/ML/Organoid/local_corr_clusters_DEGs/degs_leiden_clusters_concat.csv')

In [6]:
local_corr_modules

cluster,names,scores,logfoldchanges,pvals,pvals_adj
<int>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>
8,TCF7L2,17.26022,0.5598861,9.37471e-67,1.37851e-65
8,ATRX,16.54984,0.5173059,1.60528e-61,2.25519e-60
8,SETD5,15.57358,0.5723956,1.10065e-54,1.44599e-53
8,RBM47,14.94454,0.5174391,1.69065e-50,2.13131e-49
8,ITGA6,14.66685,0.5753797,1.05108e-48,1.29980e-47
8,NCOR1,14.65324,0.5779376,1.28446e-48,1.58654e-47
8,CHD2,14.57000,0.5525929,4.35907e-48,5.36343e-47
8,SENP6,14.30458,0.5086332,2.04868e-46,2.46633e-45
8,SBF2,14.23994,0.5610374,5.17707e-46,6.19962e-45


In [11]:
unique_clusters <- unique(local_corr_modules$cluster)

for (cluster_value in unique_clusters) {
  cluster_df <- local_corr_modules[local_corr_modules$cluster == cluster_value, ]
  
  df_name <- paste("local_corr_module_", cluster_value, sep = "")
  
  assign(df_name, cluster_df)
}

In [12]:
unique_clusters1 <- unique(local_corr_modules$cluster)

for (cluster_value in unique_clusters1) {
  df_name1 <- paste("local_corr_module_", cluster_value, sep = "")
  
  cat("Variable name:", df_name1, "\n")
}


Variable name: local_corr_module_8 
Variable name: local_corr_module_6 
Variable name: local_corr_module_3 
Variable name: local_corr_module_9 
Variable name: local_corr_module_0 
Variable name: local_corr_module_1 
Variable name: local_corr_module_2 
Variable name: local_corr_module_5 
Variable name: local_corr_module_4 
Variable name: local_corr_module_7 
Variable name: local_corr_module_10 
Variable name: local_corr_module_11 


In [23]:
local_corr_module_8

,cluster,names,scores,logfoldchanges,pvals,pvals_adj
,<int>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>
1,8,TCF7L2,17.26022,0.5598861,9.37471e-67,1.37851e-65
2,8,ATRX,16.54984,0.5173059,1.60528e-61,2.25519e-60
3,8,SETD5,15.57358,0.5723956,1.10065e-54,1.44599e-53
4,8,RBM47,14.94454,0.5174391,1.69065e-50,2.13131e-49
5,8,ITGA6,14.66685,0.5753797,1.05108e-48,1.29980e-47
6,8,NCOR1,14.65324,0.5779376,1.28446e-48,1.58654e-47
7,8,CHD2,14.57000,0.5525929,4.35907e-48,5.36343e-47
8,8,SENP6,14.30458,0.5086332,2.04868e-46,2.46633e-45
9,8,SBF2,14.23994,0.5610374,5.17707e-46,6.19962e-45


In [44]:
#gene matrix transposed
gset = list()
i = 'small'
gmt_fn = '/data/chanjlab/CRC_ZFP36L2.092023/ref/pathways_for_gsea.curated.small.110123.gmt'
gset[[i]] = gage::readList(gmt_fn)

In [71]:
RunGSEA <- function(local_corr_module_0, gset0) {
  rnk = local_corr_module_0$logfoldchanges

  names(rnk) = local_corr_module_0$names
  score = rnk[!is.na(rnk)] 
  
  fgseaRes_0 <- fgsea(pathways = gset0, 
              stats = score,
              minSize=10,
              maxSize=500)#,
              #nperm=10000)
  
  fgseaRes_0$leadingEdge = sapply(fgseaRes_0$leadingEdge, function(x) str_c(x,collapse=','))
  return(fgseaRes_0)
}

In [72]:
gsea_dir = '/data/chanjlab/CRC_ZFP36L2.092023/Patients.HTAN/output/organoid_ML/GSEA_local_corr_modules_DEGS'

In [73]:
for (mode in 'small') {
    fgseaRes_0 = RunGSEA(local_corr_module_0, gset[[mode]])

    ofile = sprintf('GSEA.logfc_local_corr_module_0.%s.txt', mode)
    write.table(fgseaRes_0, sprintf('%s/%s',gsea_dir,ofile), sep='\t', quote=F, row.names=F)

}

Warning message in fgseaMultilevel(pathways = pathways, stats = stats, minSize = minSize, :
“There were 70 pathways for which P-values were not calculated properly due to unbalanced (positive and negative) gene-level statistic values. For such pathways pval, padj, NES, log2err are set to NA. You can try to increase the value of the argument nPermSimple (for example set it nPermSimple = 10000)”


In [74]:
fgseaRes_0

pathway,pval,padj,log2err,ES,NES,size,leadingEdge
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<chr>
ABBUD_LIF_SIGNALING_1_UP,3.018480e-01,6.610090e-01,0.070789909,-0.45228800,-1.1529448,36,"ALCAM,FGFR1,PFKP,MTSS2,FOXA2,UPP1,ALDOC,CEBPB,CYB561,TGFBR2,ACVR1,PELI1,MAN1A1,ITGA6,HK2"
ACEVEDO_LIVER_CANCER_UP,NA,NA,NA,0.11013996,NA,498,"TMEM109,CHMP2B,PTTG1IP,CHCHD1,GRN,HMGB2,VAMP5,LLPH,RPF1,MARVELD2,ABHD16A,MFAP1,PAK1,MET,SCNM1,MCM4,CHMP1B,CYB561D2,ACTL6A,TYMS,STAMBPL1,HARS2,EIF1B,RPL23A,GNPAT,COMMD2,IMPA1,SRP54,TMEM183A,VBP1,SLC38A9,LINC00467,LAMTOR3,HCFC1,DYNLRB1,VPS26A,RPS27A,IRF6,KIFAP3,TMED7,RPL14,IL32,DAD1,VAMP7,MRPL13,VPS25,ORC3,PHF23,ZNF83,MRPS33,SNRPC,BOLA1,BTG3,RPLP1,PEX2,CCDC117,NFE2L2,STAG2,CASK,AQP11,HSD17B11,SMARCE1,HSPA14,XRCC1,IRAK1,RFX5,C12orf65,TMED5,HNRNPR,SMC3,ROCK2,TOX4,CNPY3,SUB1,SMNDC1,TSC22D2,PURA,ZFP36L1,TIGD5,C16orf91,ZXDB,CISD1,ABHD5,SPNS1,MRPL50,FADD,TFIP11,TALDO1,ATP6V0E1,IDI1,RNF26,NDUFB5,PON2,PRPF3,TESK1,KANK2,CKLF,RRM2B,ALG14,BZW2,BMI1,MRPS7,TATDN1,ANKMY2,SORT1,ZBTB33"
ACEVEDO_LIVER_TUMOR_VS_NORMAL_ADJACENT_TISSUE_UP,NA,NA,NA,0.12459045,NA,448,"RPS25,PTTG1IP,CHCHD1,GRN,HMGB2,DDX46,PFN1,ZMYND19,NUDT1,VAMP5,NDUFA12,LLPH,ABHD16A,SNRPF,MFAP1,SCNM1,MCM4,CHMP1B,ACTL6A,TYMS,STAMBPL1,HARS2,EIF1B,RPL23A,GNPAT,SPTSSA,C1orf198,DYNLL1,RAB22A,TMEM183A,RPL30,CHPT1,VBP1,SLC38A9,HTATSF1,DPY30,POGK,C16orf87,DYNLRB1,RPS27A,IRF6,TFRC,KIFAP3,TMED7,RPS13,SPCS2,DAD1,VAMP7,VPS25,SNRPC,BOLA1,BTG3,PEX2,CCDC117,DNAJC9,CASK,AQP11,HSD17B11,HSPA14,SLC39A1,IRAK1,DCTPP1,RFX5,CNN3,NDUFA8,UBB,TARS2,CDC42EP4,HNRNPR,SMC3,ARL1,ROCK2,CNPY3,HAUS4,PURA,MLH1,TIGD5,ZMPSTE24,ABHD5,WDR55,RPL10A,FADD,EXOSC5,TFIP11,ATP6V0E1,IDI1,RNF26,RDH10,NDUFB5,VPS28,TESK1,SNRPB,CKAP4,CKLF,ALG14,BZW2,BMI1,TFDP1,TATDN1,ANKMY2,MTX1,POLR2C,CORO1B,BTF3,TIMELESS,LSG1,LARS,SUPT4H1,CETN2,CTCF,RPL26L1,MRPS27,CDC5L,H2AFX,VPS29,IARS2,SLC9A6,CAP2,PRC1,CHMP5,YIF1B,LMNA,NOL7,TMEM167B,GPN1,OSTC,PAK1IP1,SF3B4,TIMM8A,IFRD2,ATP6V0C,TMTC4,APEX2,DCAF12,FAM120AOS,KIAA1191,GNG10"
AGUIRRE_PANCREATIC_CANCER_COPY_NUMBER_DN,9.890110e-01,1.000000e+00,0.004808996,-0.22095493,-0.5950063,121,"VLDLR,PDGFB,DUSP6,LRP2BP,PPIL6,MMP11,PIAS2,TIMP3,CSNK1E,BTG1,SFI1,IL20RA,YARS,KCTD17,AHI1,XPOT,CHEK2,WASF1"
AGUIRRE_PANCREATIC_CANCER_COPY_NUMBER_UP,1.000000e+00,1.000000e+00,0.000000000,-0.18666594,-0.5054019,158,"HAS2,ASNS,BMP4,FUT1,C5AR1,LPCAT1,SNAPC1,ZNF331,TSEN34,CASD1,PVT1,GARS,TCFL5,FAM117A,DMPK,RUVBL2,GYS1,PON3,TRRAP,PRMT1,ANXA13,ZNF787,ZNF460,EML2,DUSP3,ZNF264,DOCK4,MBOAT7"
AKT_UP_MTOR_DN.V1_DN,6.675376e-03,4.174121e-02,0.407017919,-0.56099154,-1.4748692,60,"GSDMA,TNFRSF19,IRS1,SLC18A1,MLXIPL,PROM1,PDZRN3,PHLDA2,NID1,VCAN,PSAT1,SLCO3A1,FKBP10"
AKT_UP_MTOR_DN.V1_UP,7.414830e-02,2.497221e-01,0.161978949,-0.48858697,-1.2932507,72,"TESC,DUSP10,HLA-DQB1,C3orf52,KRT23,KCNQ1,CPE,HBEGF,GLDC,BMP8B,TTLL1,SYT17,PDK3,HOXB3,DUSP8,ABCA7,RASD1,ALDOC"
ALFANO_MYC_TARGETS,2.837163e-01,6.311598e-01,0.072535186,-0.40811314,-1.0997009,119,"MAP1B,RBP1,PTPRN2,MME,KCNH2,ADM,BMP4,QSOX1,TIMP2,GLDC,MGLL,LY6E,PDLIM4,DUSP2,KIF3C,PHLDA1,SLC16A1,FYN,CDC42EP3,ASB13,RRP12,TRAP1,RUVBL1,E2F5,SLC39A6,PDIA5,HERPUD1,TARBP1,ADCY3"
ALONSO_METASTASIS_UP,9.720280e-01,1.000000e+00,0.007739273,-0.23679610,-0.6347011,98,"CA9,TUBB3,VCAN,VKORC1,TUBA1A,GOLT1A,SCD,TUBA4A"


### Loop for above

In [82]:
#Pos error- duplicates in module DEGs, and logfoldchange scores
#increase
RunGSEA <- function(local_corr_module, gset0) {
  rnk <- local_corr_module$logfoldchanges
  names(rnk) <- local_corr_module$names
  score <- rnk[!is.na(rnk)] 
  
  fgseaRes <- fgsea(pathways = gset0, 
                    stats = score,
                    minSize = 10,
                    maxSize = 500)#,
                    #nperm = 10000)
  
  fgseaRes$leadingEdge <- sapply(fgseaRes$leadingEdge, function(x) paste(x, collapse = ','))
  
  return(fgseaRes)
}

#res for each module
fgseaResults <- list()

#mod 0 -11
for (i in 0:11) {
    #name of the current local_corr_module
    module_name <- paste0("local_corr_module_", i)
    
    #GSEA for the current module
    fgseaResults[[paste0("fgseaRes_", i)]] <- RunGSEA(get(module_name), gset[['small']])
    
    #output file name
    ofile <- sprintf('GSEA.logfc_%s.txt', module_name, i)
    
    #gsea res to file
    write.table(fgseaResults[[paste0("fgseaRes_", i)]], file.path(gsea_dir, ofile), sep = '\t', quote = FALSE, row.names = FALSE)
}


Warning message in fgseaMultilevel(pathways = pathways, stats = stats, minSize = minSize, :
“There were 71 pathways for which P-values were not calculated properly due to unbalanced (positive and negative) gene-level statistic values. For such pathways pval, padj, NES, log2err are set to NA. You can try to increase the value of the argument nPermSimple (for example set it nPermSimple = 10000)”
Warning message in sprintf("GSEA.logfc_%s.txt", module_name, i):
“one argument not used by format 'GSEA.logfc_%s.txt'”
Warning message in preparePathwaysAndStats(pathways, stats, minSize, maxSize, gseaParam, :
“There are ties in the preranked stats (0.93% of the list).
The order of those tied genes will be arbitrary, which may produce unexpected results.”
Warning message in fgseaMultilevel(pathways = pathways, stats = stats, minSize = minSize, :
“There were 28 pathways for which P-values were not calculated properly due to unbalanced (positive and negative) gene-level statistic values. For such p

Warning message in sprintf("GSEA.logfc_%s.txt", module_name, i):
“one argument not used by format 'GSEA.logfc_%s.txt'”
Warning message in preparePathwaysAndStats(pathways, stats, minSize, maxSize, gseaParam, :
“There are ties in the preranked stats (1.02% of the list).
The order of those tied genes will be arbitrary, which may produce unexpected results.”
Warning message in fgseaMultilevel(pathways = pathways, stats = stats, minSize = minSize, :
“There were 37 pathways for which P-values were not calculated properly due to unbalanced (positive and negative) gene-level statistic values. For such pathways pval, padj, NES, log2err are set to NA. You can try to increase the value of the argument nPermSimple (for example set it nPermSimple = 10000)”
Warning message in fgseaMultilevel(pathways = pathways, stats = stats, minSize = minSize, :
“For some of the pathways the P-values were likely overestimated. For such pathways log2err is set to NA.”
Warning message in sprintf("GSEA.logfc_%s.txt"

In [76]:
fgseaResults[["fgseaRes_1"]]

pathway,pval,padj,log2err,ES,NES,size,leadingEdge
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<chr>
ABBUD_LIF_SIGNALING_1_UP,4.934542e-02,2.873343e-01,0.202071709,-0.5047721,-1.3613999,46,"LCN2,SCT,CAPN9,FGFR1,PSMB8,CD24,ALCAM,NMI,FBP1,CXorf56,JUNB,BCL3,IRF1,TAPBP"
ACEVEDO_LIVER_TUMOR_VS_NORMAL_ADJACENT_TISSUE_UP,1.000000e+00,1.000000e+00,0.000000000,-0.2324237,-0.6866907,492,"LCN2,TEAD2,TUBA1A,CAP2,IGFLR1,C12orf57,IRX3,PEA15,PFDN6,C8orf33,RPS21,COX7A2,CRY1,WIPI1,FOXQ1,IRAK1,MRPS21,COX6A1,EIF3J,ZMPSTE24,DDX21,TBC1D2,CES1,PSME3,EXOSC5,NDUFA7,PPIC,GRN,RNASEH2A,NHP2,ZMAT3,MRPS18C,C1orf53,PHPT1,NDUFB3,DOLK,NAA10,DYNLRB1,RPS25,PAK1IP1,SCNM1,DDX46,CDK5,MRPL36,UQCRB,NUP62,RRP36,MRPL33,TESK1,NDUFB6,RPP21,RRP15,NDUFAF2,TUBB4B,SCAMP3,CCDC167,MAN2B1,TRIM27,LAGE3,LSG1,ARSA,RPL35A,DHX16,MRPS17,SUV39H1,GNG5,EPHX1,SLC38A9,RPS13,SATB2,CD59,VPS25,HPS6,VPS28,ANKRD49,ADAT1,ZMYND19,PIP4K2C,YIPF1,EIF2B1,CDR2,TMEM199,UFC1,RRS1,MUC13,VAMP5,RPS15A,MAP1S,PDCD5,ATP6V0C,NOP10,TUSC2,SDF2L1,RPL26,LLPH,GPAA1,HS1BP3,NAT9,SRXN1,FRAT2,TCHP,FKBP5,CLN3,POGK,ETF1,GHDC,MEA1,WDR55,C1orf131,PSMD2,SAP30L,TIGD5,ITGAV,MMGT1,HIGD1A,MCTS1,PIGM,IFRD2,TIMM8A,RPS27A,NDST1,LSM10,SLC35B1,RPL30,DDAH2,KIFAP3,NEDD8,TMEM147,RPL36AL,RUVBL2,DCTPP1,CDC42EP4,CORO1B,SNRPF,DNMT1,C12orf45,RFX5,HIST2H2AC,DHX29,SUPT4H1,S100A10,GTPBP4,MCM5,KLF13,ESF1,DTYMK,AARS2,PPL,ERLEC1,TFDP1,AHCY,APEX2,CTCF,ABHD5,C1orf35,ZNF7,CHCHD1,MRPS18B,RPL23A,RANBP6,NDUFA12,UBLCP1,POLR2J3,DENND5B,TMEM183A,MAGOH,NUDT1,FADD,SET,SFXN2,NANS,SH3PXD2A,LYAR,ZMAT2,NR2C2AP,COPE,KIAA0556,MAP2K2,SRSF2,UCK2,C5orf51,SF3B4,MAPRE1,DAD1,ATG3,SURF1,CNPY3,DVL3"
AGUIRRE_PANCREATIC_CANCER_COPY_NUMBER_DN,9.980020e-01,1.000000e+00,0.002041300,-0.1838830,-0.5280511,125,"BLOC1S1,ZNF593,CACNB3,VDR,ORMDL2,RARG,SNRPD3,SMPD2,MFSD5,E2F2,POLR2F,KITLG,SLC48A1,ZBTB24,MAK16,LYPLA2,AMDHD2,HECA,ATP6V0C,QRSL1,DDX23,KCTD17,PRPF31,MTAP,SRRM1,ASCC2,PRKAG1,PFDN5,BCR,RPS6KA1,HMGN2,MECR,RANBP6,YWHAH,SCAF11,THOC5,XPC,AMD1,C12orf10,YTHDF2,SMAGP,DEPDC5,PPP5C,SGSM3,TOMM22,PISD,LSM3,TXNL4A"
AGUIRRE_PANCREATIC_CANCER_COPY_NUMBER_UP,2.587413e-01,7.455483e-01,0.077274697,-0.3720573,-1.0824347,189,"ID1,PLEKHA4,DOCK4,EXOSC4,NDUFS6,OPLAH,SIX5,GRWD1,CD3EAP,NMT1,IRF3,RPS21,DGAT1,NAPA,PSME3,TSTA3,GRN,NR1H2,NOSIP,ARFRP1,NFKBIB,BRD9,ZNHIT1,LRRC59,PRPF6,ZGPAT,NUP62,PLOD3,SPATA20,G6PC3,EPN1,MBOAT7,RCN3,ANXA13,DMPK,OPA3,STRN4,ORAI2,TLN1,ZNF444,GSDMD,ZC3H3,VPS28,SHARPIN,MRPS12,PAF1,RSAD1,NKIRAS2,ZNF580,SNRPD2,PPP2R5D,GPAA1,ASL,SLC39A4,BAX,NUCB1,U2AF2,CYHR1,AARSD1,PDK2,NME1,TPD52L2,MLX,SLC35B1,INTS1,AP4M1,UPF3A,RUVBL2,CRCP,MYL6,EIF3K,DDT,MTAP,ATXN7L1,FIS1,UCKL1,SCRIB,SARS2,ZNF787,PON2,AP2S1,ZDHHC24,MICALL2,PSMA7,PRKD2,TTF2,CREB3,NUDC"
AKT_UP_MTOR_DN.V1_DN,7.807808e-02,3.577512e-01,0.157402899,-0.4573884,-1.2709435,71,"PHGR1,TNNC2,PROM1,HPGD,NR4A3,TYMP,GSDMA,CAP2,ERF,SLC18A1,IL13RA1,RPS6KB2,RTKN,SNHG6,AMPD2,RNF25,SLCO3A1,MAZ,BCS1L"
AKT_UP_MTOR_DN.V1_UP,6.587590e-03,7.220516e-02,0.407017919,-0.5090181,-1.4176253,76,"CXCL1,EDN1,KRT23,CDKN2B,TMCC3,HLA-DQB1,DUSP10,CPE,KCNQ1,RASD1,CD55,DUSP1,GIPC1,ATF3,MVP,PARM1,CTNS,NFKB2,JUNB,TPRN,KLF6,GCH1,MAN2B2,ANXA13,NOL12"
ALFANO_MYC_TARGETS,9.490509e-02,4.019087e-01,0.141225120,-0.4113927,-1.1846568,137,"ID3,EMP1,SMPD1,RBP1,LY6E,CSRP1,DUSP2,MME,PLEC,CLCN7,SYNGR2,TOMM40,NOP56,ABCF2,VCL,DDX21,MVP,SURF2,PDCD11,CDX1,TCOF1,IMPDH1,ITGA7,SDC1,GPR143,NBL1,IRF1,CLCN2,FARSA,PPP1R15A,MGLL,CD59,RRP12,HS3ST1,EIF2B1,AATF,MYCBP,FYN,TTLL12,AFAP1,ATP8A1,TRIP6,TCF19,LTBP1,GNAI2,SLC20A1,FKBP4,PAFAH1B3,DNAJC11"
ALONSO_METASTASIS_UP,4.595405e-01,9.842265e-01,0.049490487,-0.3558761,-1.0188776,117,"EMP1,COX6B1,TUBA1A,UBL5,CDKN1A,SEMA5A,ATP6V0D1,TIMM10,RPS10,AKT1S1,HSPA8,TP53I3,RCBTB2,HSPB1,COX8A,NDUFB11,NHP2,DHRS11,NDUFB3,PRDX5"
ASC_SMITH_CELL2018,7.435367e-01,1.000000e+00,0.028574398,-0.3229660,-0.8276416,25,"VSNL1,DDX46,TCOF1,SLCO3A1"


#### Save significant pthwys

In [96]:
#loop for saving all sig pathways

#store the filtered dataframes
filtered_dfs <- list()

#loop through each module and filter the dataframe
for (i in 0:11) {
    #dataframe for the current module
    fgsea_res <- fgseaResults[[paste0("fgseaRes_", i)]]
    gsea_df <- data.frame(fgsea_res)
    
    # Filter the dataframe based on conditions
    filtered_df <- gsea_df %>% dplyr::filter(abs(NES) > 1 & padj < 0.05)
    
    # Save the filtered dataframe as a variable
    assign(paste0("gsea_df_fgseaRes_", i, "_save"), filtered_df, envir = .GlobalEnv)
    
    # Add the filtered dataframe to the list
    filtered_dfs[[i + 1]] <- filtered_df
    
    # Define the file paths for saving
    txt_file <- sprintf("/data/chanjlab/CRC_ZFP36L2.092023/Patients.HTAN/output/organoid_ML/GSEA_local_corr_modules_DEGS/module_significant_pathways/GSEA_sig_pthwys_local_corr_module_%s.txt", i)
    csv_file <- sprintf("/data/chanjlab/CRC_ZFP36L2.092023/Patients.HTAN/output/organoid_ML/GSEA_local_corr_modules_DEGS/module_significant_pathways/GSEA_sig_pthwys_local_corr_module_%s.csv", i)
    
    # Save the filtered dataframe as text and CSV files
    write.table(filtered_df, file = txt_file, sep = "\t", quote = FALSE, row.names = FALSE)
    write.csv(filtered_df, file = csv_file, quote = FALSE, row.names = FALSE)
}

# Optionally, you can access the saved dataframes using gsea_df_fgseaRes_#_save


In [105]:
gsea_df_fgseaRes_5_save

pathway,pval,padj,log2err,ES,NES,size,leadingEdge
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<chr>
BENPORATH_EED_TARGETS,1.575756e-04,0.0086666556,0.5188481,-0.6383356,-1.610327,156,"HOXD3,ATOH1,SLC22A3,MAGI2,RIN3,HOXA13,PGM5,PRKG1,BMPR1B,SIDT1,ITPKA,EML5,CYP1B1,DOCK11,TRIM9,CXCL16,GDA,FBP1,COL27A1,FOXD2,MSX2,PIP5K1B,REPS2,NPNT,PXMP2,PPM1E,SOX5,CA3"
BENPORATH_ES_WITH_H3K27ME3,1.035101e-05,0.0011386106,0.5933255,-0.7137064,-1.784994,117,"HOXD3,ATOH1,SLC22A3,HOXA13,CRIP1,AMN,C1orf115,PGM5,RAMP1,TLE2,PRKD1,PRKG1,NBL1,SIDT1,ITPKA,PRKAG2,TRIM9,SLC40A1,CXCL16,ABCC3,FBP1,XYLT1,BLVRB,COL27A1,FOXD2,MSX2,PIP5K1B,REPS2,NPNT,PXMP2,PPM1E"
BENPORATH_SUZ12_TARGETS,4.320543e-06,0.0007920995,0.6105269,-0.6955469,-1.750085,134,"HOXD3,ATOH1,RIN3,PTCHD1,RASEF,CRIP1,AMN,GIPC2,C1orf115,GNAS,PGM5,ST3GAL6,PRKG1,NBL1,SIDT1,LAMA3,ITPKA,EML5,CYP1B1,TRIM9,SLC40A1,CXCL16,GDA,FBP1,XYLT1,FOXA1,COL27A1,FOXD2,PIP5K1B,REPS2,NPNT,PXMP2,PPM1E,PCDH11X,CA3,MGLL,ACSS1,NDRG1,RARRES1,MT1X,IHH,NHS,OSR2,DDAH1,TET2,MNX1,TMEM30B,DNAJC22,CAMK2N1"
EED_TARGETS_BENPORATH,1.349032e-04,0.0082440867,0.5188481,-0.6614579,-1.664505,144,"HOXD3,ATOH1,SLC22A3,MAGI2,RIN3,HOXA13,PGM5,PRKG1,BMPR1B,SIDT1,ITPKA,EML5,CYP1B1,DOCK11,TRIM9,CXCL16,GDA,FBP1,COL27A1,FOXD2,MSX2,PIP5K1B,REPS2,NPNT,PXMP2,PPM1E,SOX5,CA3"
GO_POSITIVE_REGULATION_OF_NEURON_DIFFERENTIATION,4.349771e-05,0.0039872898,0.5573322,-0.7404586,-1.844364,77,"HOXD3,ATOH1,MAGI2,ROBO2,RIMS2,CNR1,SERPINE2,RARB,PRKD1,NBL1,ITPKA,AKAP5"
GO_REGULATION_OF_NEURON_DIFFERENTIATION,7.006538e-04,0.0321132986,0.4772708,-0.6311270,-1.589002,138,"HOXD3,ATOH1,MAGI2,ROBO2,RIMS2,CNR1,SERPINE2,RARB,PRKD1,BCL11B,NBL1,ITPKA,SLC6A4,ID1,NOTCH3,AKAP5,XK,FOXA1,TGIF2,DLL1"
H3K27_BOUND_BENPORATH,2.699958e-06,0.0007424883,0.6272567,-0.7309848,-1.829518,109,"HOXD3,ATOH1,SLC22A3,HOXA13,CRIP1,AMN,PGM5,RAMP1,TLE2,PRKD1,PRKG1,NBL1,SIDT1,ITPKA,PRKAG2,TRIM9,SLC40A1,CXCL16,ABCC3,FBP1,XYLT1,BLVRB,COL27A1,FOXD2,MSX2,PIP5K1B,REPS2,NPNT,PXMP2,PPM1E"
HOLLEMAN_ASPARAGINASE_RESISTANCE_B_ALL_UP,7.978772e-05,0.0058553627,0.5384341,0.6673830,2.808364,12,"TPM4,EIF3L,FBL,RPL4,RPL3,EEF1G,EIF3K,EIF3D,RPL7A,RPS3"
HotSpotModule_3.LIPID_METABOLISM,1.494485e-06,0.0007424883,0.6435518,-0.9120829,-2.135753,23,"ADH1C,HPGD,AKR1B10,ZG16,FABP2,APOA1,TM4SF20,RAMP1,SLC51B,MEP1A,ALDH1A1,ANXA13,SDCBP2,SLC40A1"


In [101]:
read_module_0 = read.table("/data/chanjlab/CRC_ZFP36L2.092023/Patients.HTAN/output/organoid_ML/GSEA_local_corr_modules_DEGS/module_significant_pathways/GSEA_sig_pthwys_local_corr_module_0.txt", header = TRUE, sep = '\t')

In [102]:
read_module_0

pathway,pval,padj,log2err,ES,NES,size,leadingEdge
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<chr>
BENPORATH_EED_TARGETS,5.303902e-05,9.185600e-04,0.5573322,-0.5377383,-1.460445,189,"FZD10,PLXND1,COL4A5,MKX,SHISA6,ANKRD18A,MAGI2,FOXQ1,PPM1E,RASSF5,NRCAM,INHBB,STC2,SLCO4A1,TMOD2,ADM,SLC22A3,NEUROD1,KCNQ1,SOX21,EPHB3,PGM5,FOXA2,PCSK1N,CD44,GAD1,SETD7,GABRA2,ELAVL2,DPY19L2,HOXA3,GADD45G,SGPP2,BAIAP2,DMRT2,DUSP4,RIMKLA,DLL4,SLC1A4,MT1A,NFIX,HLA-A,CSNK1E,NAV2,HOXB3,SIM2,TUBA4A,KIAA1324,IL20RA,EFNA3,GRK5,CASZ1,PIR,LGR5,ABTB2,TBCB,EGR3,HHLA3,GALNT10,STK17A,B4GALT6,RGS10,ATF3,SOX5,HYOU1,PPP1R10,ZNF503,IER5L,INTU"
BENPORATH_ES_WITH_H3K27ME3,4.946945e-10,6.406294e-08,0.8012156,-0.6363499,-1.727339,155,"FZD10,SMOC2,COL4A5,MKX,EPHA4,SHISA6,ANKRD18A,PTPRN2,NOTUM,NPAS2,FOXQ1,PPM1E,SLC25A27,RASSF5,MLXIPL,STC2,GRB10,SEMA4F,TMOD2,DUSP6,SLC22A3,ADAMTS17,NEUROD1,KCNQ1,LEF1,PRUNE2,EPHB3,PGM5,FOXA2,CYP39A1,DTNB,PARP8,GABRA2,ELAVL2,RAPGEF4,DPY19L2,HOXA3,SLCO3A1,SGPP2,DMRT2,ADAM15,DUSP4,RIMKLA,DLL4,SLC1A4,MT1A,NFIX,HLA-A,SKAP1,NAV2,HOXB3,TTLL7,SYT7,DUSP8,SIM2,CBX4,TNFRSF1B,LAMB1,RPS6KA2,DYNC1I1,KIAA1324,OSBP2,EFNA3,GRK5,CASZ1,PIR,LGR5,ABTB2"
BENPORATH_SUZ12_TARGETS,1.617145e-06,5.767586e-05,0.6435518,-0.5864793,-1.588826,151,"FZD10,COL4A5,MKX,EPHA4,SHISA6,ANKRD18A,NPAS2,PPM1E,RASSF5,NRCAM,CDH13,BHMT,TLL2,SEMA4F,SLCO4A1,TMOD2,ADM,DUSP6,NEUROD1,SOX21,LEF1,GRIN2B,NKD1,EPHB3,PGM5,FOXA2,PCSK1N,CD44,PYCARD,SETD7,GABRA2,RAPGEF4,DPY19L2,GADD45G,SGPP2,DMRT2,MGLL,DUSP4,RIMKLA,DLL4,SLC1A4,MT1A,NFIX,LRRC8C,NAV2,HOXB3,SIM2,NHS,TNFRSF1B,TUBA4A,KIAA1324,EFNA3,CASZ1,PIR,LGR5,EFNB1,ABTB2"
CREIGHTON_ENDOCRINE_THERAPY_RESISTANCE_2,2.478207e-07,1.426346e-05,0.6749629,-0.5946224,-1.610288,165,"C4orf47,MKX,ARMC4,NPHP1,DNALI1,CCDC40,JAZF1,PPM1E,DNAH6,SPATA6,TMEM231,TMEM67,LRP2BP,DNAH7,CEP112,PPIL6,WDR66,TSPAN2,DNAH11,ABLIM2,ARL6,IQCH,RSPH1,C5AR1,KIF3A,HES6,C3orf67,SLC41A1,DPCD,C16orf46,APOLD1,EML6,FANK1,FAM149A,EML1,CCNO,WDR78,LRRC10B,TTC12,IFT81,NGEF,WDR63,NPHP4,MAK,EFHC1,KLHDC9,TRIM73,MOK,PCSK6,CCDC30,SRGAP3,PHTF1,LRRC49,NEK2,KIF9,FSD1L,CDKL3,CCP110,LRRC23,TTC26,LRWD1,RPGRIP1L,DNAH12,IQCG,IQCK,ODF2,CRY1,SEC61A2,CLUAP1,TMCC1,MORN2,NETO2,DNAL4,AKNA,COG7,ARMC2,REEP1"
EED_TARGETS_BENPORATH,6.937626e-05,1.063341e-03,0.5384341,-0.5444007,-1.478530,179,"FZD10,PLXND1,COL4A5,MKX,ANKRD18A,MAGI2,FOXQ1,PPM1E,RASSF5,NRCAM,INHBB,STC2,SLCO4A1,TMOD2,ADM,SLC22A3,NEUROD1,KCNQ1,SOX21,EPHB3,PGM5,FOXA2,PCSK1N,CD44,GAD1,SETD7,GABRA2,ELAVL2,DPY19L2,HOXA3,GADD45G,SGPP2,BAIAP2,DMRT2,DUSP4,DLL4,SLC1A4,MT1A,NFIX,HLA-A,CSNK1E,NAV2,HOXB3,SIM2,TUBA4A,KIAA1324,IL20RA,EFNA3,GRK5,CASZ1,PIR,LGR5,ABTB2,TBCB,EGR3,HHLA3,GALNT10,STK17A,B4GALT6,RGS10,ATF3,SOX5,HYOU1,PPP1R10,ZNF503,IER5L,INTU"
GO_AXON,1.507739e-04,1.952521e-03,0.5188481,-0.5496478,-1.482773,141,"ROBO1,CADM2,TUBB3,CORO1A,ADRA2C,DRD2,EPHA4,PTPRN2,STXBP1,MME,RAB3A,NFASC,ALCAM,GABBR1,NRCAM,PTPRO,UCHL1,DISC1,DNM3,AVIL,SEMA3A,GABRA2,SACS,TIAM1,SCN8A,SYT7,TNFRSF1B,LRP8,SRCIN1,NCS1,HOMER1,LMTK3,KATNB1,GARS,GRIK2,PALLD,CHRM3"
GO_AXONEME_ASSEMBLY,6.524115e-03,4.224364e-02,0.4070179,-0.7592426,-1.633731,11,"ARMC4,CCDC40,DNAH7,RSPH1,TTLL1,SPAG1,LRRC6"
GO_AXON_PART,6.749961e-03,4.316642e-02,0.4070179,-0.5560626,-1.459647,69,"ADRA2C,DRD2,EPHA4,PTPRN2,STXBP1,RAB3A,NFASC,GABBR1,NRCAM,UCHL1"
GO_CELL_CELL_ADHESION,3.274336e-03,2.458124e-02,0.4317077,-0.5026365,-1.360831,149,"ROBO1,CADM2,NPHP1,CD74,CADM1,STXBP1,CLDN18,NFASC,ALCAM,NRCAM,CDH13,SLC7A11,SDK1,JAG2,LEF1,CD44,RORA,FGFRL1,NRXN3,CEACAM1,DLL4,MMP24,NPHP4,PODXL2,DPP4,FYN,ANXA9,LAMB1,LIG4,CELSR1"


### Lollipop Plots

In [88]:
gsea_df_fgseaRes_1 = data.frame(fgseaResults[["fgseaRes_1"]])

In [89]:
gsea_df_fgseaRes_1 = gsea_df_fgseaRes_1[!duplicated(gsea_df_fgseaRes_1$pathway),]

In [ ]:
gsea_df_